In [1]:
# Install Requirements

!pip install torch pandas numpy scipy scikit-learn pyyaml

In [2]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Light_CGN_project/data/movielens/latest_small/ratings.csv"

Mounted at /content/drive


In [5]:
# Import Libraries

import sys
sys.path.append('/content/lightgcn-res')

import torch
import pandas as pd
import numpy as np
import yaml

from src.model import LightGCN
from src.dataset import load_movielens
from src.metrics import precision_at_k

ModuleNotFoundError: No module named 'src'

In [ ]:
# Load Configuration

with open("../configs/lightgcn.yaml") as f:
    config = yaml.safe_load(f)

print(config)

In [ ]:
# Load Dataset

ratings, num_users, num_items = load_movielens(DATA_PATH)

print("Users:", num_users)
print("Items:", num_items)

ratings.head()

In [ ]:
# Initialize Model

model = LightGCN(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=config["embedding_dim"]
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)

print(model)

In [ ]:
# Prepare Training Data

users = torch.tensor(ratings["userId"].values)
items = torch.tensor(ratings["itemId"].values)

In [ ]:
# Training Loop

epochs = config["epochs"]

for epoch in range(epochs):

    optimizer.zero_grad()

    user_emb, item_emb = model()

    scores = (user_emb[users] * item_emb[items]).sum(dim=1)

    loss = -torch.log(torch.sigmoid(scores)).mean()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

In [ ]:
# Generate Recommendations

user_id = 1

user_vector = model.user_embedding.weight[user_id]

scores = torch.matmul(
    model.item_embedding.weight,
    user_vector
)

topk = torch.topk(scores, config["topk"])

print("Recommended items:", topk.indices)

In [ ]:
# Evaluate Precision@K

recommended = topk.indices.numpy()
relevant = ratings[ratings["userId"] == user_id]["itemId"].values

precision = precision_at_k(recommended, relevant, config["topk"])

print("Precision@K:", precision)

In [ ]:
# Save Model

MODEL_PATH = "/content/drive/MyDrive/recsys-results/lightgcn_model.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved")